### Packages for the Data Generation and Mopdelling of PD (Probability of Default), LGD (Loss Given Default) and EAD (Exposure at Default)

In [15]:
# Data Management and Processing
import pandas as pd
import numpy as np
import scipy
import random

In [16]:
# Machine Learning and Statistics
import sklearn
import statsmodels.api as sm
import tensorflow as tf

### Data generator

##### Support functions

In [17]:
############# Function to create a profession based on the educational level #########################
def generate_profession(education):
    if education == "high school or lower":
        return random.choices(["LowSkilled", "Unemployed_LowSkilled"], weights=[0.9, 0.1], k=1)[0]
    if education == "ausbildung":
        return random.choices(["MediumSkilled", "Unemployed_MediumSkilled"], weights=[0.9, 0.1], k=1)[0]
    if education in ["bachelor degree", "post graduate degree"]:
        return random.choices(["HighSkilled", "Unemployed_HighSkilled"], weights=[0.9, 0.1], k=1)[0]

In [22]:
############################### Function to generate monthly income and expenditure ##################################                

# Define income parameters for different profession levels and age ranges
income_parameters = {
    ("LowSkilled", "Unemployed_LowSkilled"): {
        (30, 35): {"mean": 1500, "std_dev": 200, "max_income": 3000},
        (36, 40): {"mean": 1800, "std_dev": 200, "max_income": 3300},
        (41, 45): {"mean": 2000, "std_dev": 250, "max_income": 3600},
        (46, 50): {"mean": 2200, "std_dev": 300, "max_income": 4000},
        (51, 55): {"mean": 2400, "std_dev": 350, "max_income": 4200},
        (56, 60): {"mean": 2600, "std_dev": 400, "max_income": 4500},
        (61, 65): {"mean": 2800, "std_dev": 600, "max_income": 5000},
    },
    ("MediumSkilled", "Unemployed_MediumSkilled"): {
        (30, 35): {"mean": 1800, "std_dev": 300, "max_income": 4000},
        (36, 40): {"mean": 2300, "std_dev": 400, "max_income": 5000},
        (41, 45): {"mean": 2600, "std_dev": 500, "max_income": 6000},
        (46, 50): {"mean": 3000, "std_dev": 600, "max_income": 7000},
        (51, 55): {"mean": 3500, "std_dev": 800, "max_income": 8000},
        (56, 60): {"mean": 4000, "std_dev": 800, "max_income": 9000},
        (61, 65): {"mean": 5000, "std_dev": 1000, "max_income": 10000},
    },
    ("HighSkilled", "Unemployed_HighSkilled"): {
        (30, 35): {"mean": 3000, "std_dev": 500, "max_income": 10000},
        (36, 40): {"mean": 4500, "std_dev": 700, "max_income": 15000},
        (41, 45): {"mean": 6000, "std_dev": 1000, "max_income": 20000},
        (46, 50): {"mean": 7000, "std_dev": 1500, "max_income": 25000},
        (51, 55): {"mean": 8000, "std_dev": 2000, "max_income": 30000},
        (56, 60): {"mean": 9000, "std_dev": 3000, "max_income": 40000},
        (61, 65): {"mean": 10000, "std_dev": 4000, "max_income": 50000},
    }
}

# To calculate the monthly income and expenditure
def generate_income_expense(profession_undertake, current_age, num_dependents):
    # Iterate over income parameters for each profession group
    for profession_group, age_ranges in income_parameters.items():
        # Check if profession_undertake is one of the professions in the profession_group tuple
        if profession_undertake in profession_group:
            # Iterate over the age ranges and income parameters
            for age_range, params in age_ranges.items():
                if age_range[0] <= current_age <= age_range[1]:
                    mean_income = params["mean"]
                    std_dev = params["std_dev"]
                    max_income = params["max_income"]
                    unemployement_money = mean_income * 0.5  # 50% of mean income for unemployment

                    # If profession_undertake contains the word "Unemployed" before "_", return the unemployment money
                    if profession_undertake.split("_")[0] == "Unemployed":
                        income = unemployement_money
                        expenditure = generate_expenditure(income, mean_income, num_dependents) # The belong to the population under mean_income
                        return income, expenditure
                    
                    # If profession_undertake does not contain the word "Unemployed", generate income with an specific rule
                    else:
                        calc_income = int(np.random.normal(mean_income, std_dev))
                        income = max(unemployement_money, min(calc_income, max_income))  
                        expenditure = generate_expenditure(income, mean_income, num_dependents)
                        return income, expenditure

# Support function to calculate the expenditure
def generate_expenditure(income, mean_income, num_dependents):
    # Values for low and for high income people (lower possible value, mode, higher possible value) depending on the number of dependents
    low_income_params = [(0.5, 0.7, 1.5), (0.7, 0.8, 1.5), (0.8, 0.9, 1.5), (0.9, 0.9, 1.5), (0.9, 1.0, 1.5)]
    high_income_params = [(0.5, 0.7, 1.5), (0.6, 0.7, 1.5), (0.7, 0.75, 1.5), (0.7, 0.75, 1.5), (0.8, 0.85, 1.5)]
    # The parameters that should be taken depend on whether the income is below or above the mean income
    params = low_income_params if income < mean_income else high_income_params
    # Here we recover the parameters
    left, mode, right = params[min(num_dependents, 4)]  # Ensure index stays within range
    # The expenditure is calculated given a rule of min, mode, max
    expenditure = income * np.random.triangular(left=left, mode=mode, right=right)
    return expenditure



In [ ]:
############################### Function to generate the credit ##################################

In [ ]:
############################### Function to generate the Y-variable (default 0, 1) ##################################     
def default_y_calculation(profession, debt_income_ratio):
    # past_credits, dependents, professon, debt_income_ratio
    if profession == ("Unemployed_LowSkilled" or "Unemployed_MediumSkilled") and debt_income_ratio > 0.1:
        return 1
    if profession == ("Unemployed_HighSkilled") and debt_income_ratio > 0.2:
        return 1     


##### Data Generator for the original state of individuals

In [ ]:
# Function to generate data accordingly to some requirements
def data_generator(number_of_customers):
    data = []
    for i in range(number_of_customers):
        
        # ---------------- X-Variables -------------------------------#
        ##### Variables not directly dependent on other variables #####
        name = f"name{i}" # names are created according to the index "i"
        age = random.randint(30, 60) # As the maximum attainable age that we want in the game is 65
        education_level = random.choices(["high school or lower", "ausbildung", "bachelor degree", "post graduate degree"],  weights=[0.3, 0.3, 0.3, 0.1], k=1)[0]
        # Number of unpaid past credits
        past_credits = random.choices([0, 1, 2, 3], weights=[0.6, 0.3, 0.08, 0.02], k=1)[0] # This emphasizes 0 and 1 unpaid credits
        # Number of dependents
        dependents = random.choices([0, 1, 2, 3, 4], weights=[0.6, 0.3, 0.06, 0.03, 0.01], k=1)[0] # This emphasizes 0 and 1 unpaid credits
        
        ##### Variables directly dependent on other variables #####
        # Generate profession based on education level
        profession = generate_profession(education_level)
        # Generate monthly income based on profession, age and number of dependents
        monthly_income = generate_income_expense(profession, age, dependents)[0]
        # Generate monthly expenditure dependening on the income, mean income and number of dependents
        monthly_expenditure = generate_income_expense(profession, age, dependents)[1]
        
        ##### Other variables generated from the variables above #####
        savings_debt = monthly_income - monthly_expenditure
        # Debt to income ratio
        if savings_debt < 0:
            #debt_to_income_ratio = f"{abs(savings_debt/monthly_income):.2%}"
            debt_to_income_ratio = abs(savings_debt/monthly_income)
        else:
            #debt_to_income_ratio = f"{0:.2%}"
            debt_to_income_ratio = 0
        
        # ---------------- Credit amount requested and time of the request--------------------------------#
        
        
        # ---------------- Y-Variable --------------------------------#
        
        data.append({
            'name': name,
            'age': age,
            'educational level': education_level,
            'number of not paid past credits': past_credits,
            'dependents': dependents,
            'profession': profession,
            'monthly income': monthly_income,
            'monthly expenditure': monthly_expenditure,
            'savings (debt)': savings_debt,
            'debt-to-income ratio': debt_to_income_ratio
        })

    # Create a pandas DataFrame
    df = pd.DataFrame(data)
    return df

##### Generating one data frame

In [42]:
number_of_customers_1 = 1000
df_1 = data_generator(number_of_customers_1)
df_1

,name,age,educational level,number of not paid past credits,dependents,profession,monthly income,monthly expenditure,savings (debt),debt-to-income ratio
0,name0,30,bachelor degree,0,0,HighSkilled,2775.0,2355.694108,419.305892,0.000000
1,name1,60,high school or lower,1,0,LowSkilled,2717.0,2252.037918,464.962082,0.000000
2,name2,43,ausbildung,0,0,MediumSkilled,1890.0,1559.479325,330.520675,0.000000
3,name3,58,ausbildung,1,0,MediumSkilled,2616.0,3598.837038,-982.837038,0.375702
4,name4,39,ausbildung,0,1,MediumSkilled,2721.0,2851.641482,-130.641482,0.048012
...,...,...,...,...,...,...,...,...,...,...
995,name995,60,ausbildung,1,1,MediumSkilled,3250.0,2741.700839,508.299161,0.000000
996,name996,54,ausbildung,0,0,Unemployed_MediumSkilled,1750.0,1229.807364,520.192636,0.000000
997,name997,52,bachelor degree,0,0,HighSkilled,5991.0,7098.386856,-1107.386856,0.184842
998,name998,50,high school or lower,0,1,Unemployed_LowSkilled,1100.0,947.634884,152.365116,0.000000


##### DF statistics

In [43]:
df_1.describe()

,age,number of not paid past credits,dependents,monthly income,monthly expenditure,savings (debt),debt-to-income ratio
count,1000.000000,1000.0000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,45.015000,0.5200,0.529000,3706.588000,3495.486011,211.101989,0.125927
std,8.946441,0.7281,0.774441,2472.470159,2511.651833,1660.273553,0.260592
min,30.000000,0.0000,0.000000,750.000000,558.874693,-10080.234890,0.000000
25%,37.750000,0.0000,0.000000,1976.750000,1783.490411,-361.209532,0.000000
50%,45.000000,0.0000,0.000000,2781.500000,2614.320193,224.597128,0.000000
75%,53.000000,1.0000,1.000000,4761.500000,4462.414675,822.277990,0.144117
max,60.000000,3.0000,4.000000,16122.000000,15208.234890,11843.901754,2.001882


Monthly income by profession

In [45]:
df_1.groupby('profession')['monthly income'].describe()

,count,mean,std,min,25%,50%,75%,max
profession,,,,,,,,
HighSkilled,361.0,6110.678670,2560.290955,1887.0,4278.00,5802.0,7503.0,16122.0
LowSkilled,280.0,2098.614286,466.770305,1219.0,1733.75,2028.0,2427.5,3513.0
MediumSkilled,276.0,2762.938406,909.622431,1149.0,2063.75,2627.5,3307.0,5455.0
Unemployed_HighSkilled,26.0,3144.230769,1140.554858,1500.0,2250.00,3250.0,4000.0,4500.0
Unemployed_LowSkilled,27.0,1088.888889,195.789002,750.0,950.00,1200.0,1250.0,1300.0
Unemployed_MediumSkilled,30.0,1310.000000,351.695401,900.0,900.00,1300.0,1500.0,2000.0


Debt-to-income ratio by profession

In [46]:
df_1.groupby('profession')['debt-to-income ratio'].describe()

,count,mean,std,min,25%,50%,75%,max
profession,,,,,,,,
HighSkilled,361.0,0.147655,0.295134,0.0,0.0,0.0,0.175672,2.001882
LowSkilled,280.0,0.084267,0.172518,0.0,0.0,0.0,0.074085,0.914748
MediumSkilled,276.0,0.155278,0.307294,0.0,0.0,0.0,0.177080,1.726779
Unemployed_HighSkilled,26.0,0.065335,0.103326,0.0,0.0,0.0,0.125828,0.389400
Unemployed_LowSkilled,27.0,0.080895,0.122342,0.0,0.0,0.0,0.139769,0.449369
Unemployed_MediumSkilled,30.0,0.076287,0.122730,0.0,0.0,0.0,0.115516,0.424219
